# 03 — Rope Database Exploration

ropesim ships with a comprehensive rope database. This notebook shows how to query it, compare specs across ropes, and visualise the trade-offs between diameter, weight, impact force, and fall rating.

In [ ]:
import ropesim.notebook
from ropesim import Rope, RopeDatabase, RopeType
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

## Load the full database

In [ ]:
db = RopeDatabase()
all_names = list(db.names())
print(f'Total ropes in database: {len(all_names)}')
print('\nAll ropes:')
for n in sorted(all_names):
    print(f'  {n}')

## Load all ropes and build a comparison DataFrame

In [ ]:
ropes = [Rope.from_db(n) for n in all_names]

# Extract key spec fields into lists
names     = [r.spec.name for r in ropes]
diameters = [r.spec.diameter_mm for r in ropes]
weights   = [r.spec.weight_gpm for r in ropes]
impacts   = [r.spec.impact_force_kn for r in ropes]
falls     = [r.spec.number_of_falls for r in ropes]
dyn_elong = [r.spec.dynamic_elongation_pct for r in ropes]
rope_type = [r.spec.rope_type.value for r in ropes]

print(f'Diameter range:     {min(diameters):.1f} – {max(diameters):.1f} mm')
print(f'Weight range:       {min(weights):.0f} – {max(weights):.0f} g/m')
print(f'Impact force range: {min(impacts):.1f} – {max(impacts):.1f} kN')
print(f'Fall rating range:  {min(falls)} – {max(falls)} falls')

## Scatter: diameter vs impact force

Thinner ropes are generally lighter but can have higher impact forces because they stretch more abruptly.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
type_colours = {'single': '#2270c4', 'half': '#27ae60', 'twin': '#e74c3c'}

for rt in set(rope_type):
    idx = [i for i, t in enumerate(rope_type) if t == rt]
    ax.scatter([diameters[i] for i in idx], [impacts[i] for i in idx],
               color=type_colours.get(rt, 'grey'), label=rt, s=60, alpha=0.8)

# Label each point
for name, d, imp in zip(names, diameters, impacts):
    short = name.split()[0][:8]
    ax.annotate(short, (d, imp), fontsize=7, alpha=0.7,
                xytext=(3, 2), textcoords='offset points')

ax.axhline(8.5, color='red', linestyle='--', linewidth=0.8, label='8.5 kN (single limit)')
ax.axhline(12.0, color='orange', linestyle='--', linewidth=0.8, label='12 kN (half limit)')
ax.set_xlabel('Diameter (mm)')
ax.set_ylabel('Impact force (kN)')
ax.set_title('Rope diameter vs impact force')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Bar chart: weight vs number of UIAA falls

A rope with fewer falls isn't necessarily weaker — it may just be very light (less material = absorbs falls harder on each catch).

In [ ]:
# Sort by weight
order = sorted(range(len(names)), key=lambda i: weights[i])
sorted_names = [names[i].replace(' ', '\n') for i in order]
sorted_weights = [weights[i] for i in order]
sorted_falls_v = [falls[i] for i in order]

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()

bars = ax1.bar(sorted_names, sorted_weights, color='#5b9bd5', alpha=0.7, label='Weight (g/m)')
ax2.plot(sorted_names, sorted_falls_v, 'D-', color='#e74c3c', markersize=6, label='UIAA falls')

ax1.set_ylabel('Weight (g/m)', color='#2270c4')
ax2.set_ylabel('UIAA fall rating', color='#e74c3c')
ax1.set_title('Rope weight vs UIAA fall rating (sorted by weight)')
ax1.tick_params(axis='x', labelsize=7)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8)
plt.tight_layout()
plt.show()

## Side-by-side full comparison table

Print a formatted table of all specs.

In [ ]:
header = f'{"Name":<35} {"Type":<7} {"Diam":>6} {"g/m":>5} {"kN":>5} {"Elong%":>7} {"Falls":>6}'
print(header)
print('-' * len(header))
for r in sorted(ropes, key=lambda x: x.spec.diameter_mm):
    s = r.spec
    print(f'{s.name:<35} {s.rope_type.value:<7} {s.diameter_mm:>5.1f}mm '
          f'{s.weight_gpm:>4.0f} {s.impact_force_kn:>5.2f}kN '
          f'{s.dynamic_elongation_pct:>6.1f}% {s.number_of_falls:>5}')

## Rope retirement calculator

EN 892 rates ropes by number of UIAA fall tests (standardised falls at FF 1.772). Real-world use degrades ropes faster. A rough field estimate: each severe lead fall consumes ~0.5–1 UIAA fall rating units.

In [ ]:
def retirement_estimate(rope_name: str, falls_taken: int, condition: str = 'good') -> None:
    rope = Rope.from_db(rope_name)
    rating = rope.spec.number_of_falls
    # Degradation factor by condition
    factor = {'new': 0.5, 'good': 0.8, 'worn': 1.2, 'damaged': 2.0}.get(condition, 1.0)
    consumed = falls_taken * factor
    remaining = max(rating - consumed, 0)
    pct = 100 * remaining / rating
    status = 'RETIRE' if pct < 20 else ('CHECK' if pct < 50 else 'OK')
    print(f'{rope_name}')
    print(f'  Rating: {rating} UIAA falls | Falls taken: {falls_taken} | Condition: {condition}')
    print(f'  Estimated remaining capacity: {remaining:.1f} falls ({pct:.0f}%) --> {status}')
    print()

retirement_estimate('Beal Opera 8.5 Dry', falls_taken=8, condition='good')
retirement_estimate('Mammut Serenity 8.9 Dry', falls_taken=3, condition='worn')
retirement_estimate('Sterling Evolution Dry 9.4', falls_taken=12, condition='good')

## Key takeaways

- Thinner / lighter ropes tend to have **higher impact forces** — there is less material to absorb energy, so each fall is harder on the climber and anchor.
- UIAA fall rating is a **minimum standard**, not a fall counter. Real ropes often outlast their rating in normal climbing use.
- **Dynamic elongation** is a key comfort metric — higher elongation = softer catch, but also means you fall further.